# Scraping Reviews for Tunisian Telecom Apps

This notebook is designed to collect user reviews from the Google Play Store for three major Tunisian telecom providers: **Orange**, **Ooredoo**, and **Tunisie Telecom (MyTT)**. The goal is to gather raw data for further analysis, such as sentiment analysis and churn prediction.

---

## Workflow Overview

### 1. **Setup**
- We define the app IDs for the three telecom providers. These IDs are used to locate the apps on the Google Play Store.
- A folder structure is created to store the raw data (`data/01_raw/`).

### 2. **Scraping Reviews**
- A function `scrape_app` is implemented to:
    - Fetch all reviews for a given app.
    - Filter and keep only relevant columns like review content, score, and date.
    - Tag the reviews with the app's name and source.
- The reviews are fetched in **French** and **Arabic**, as these are the primary languages used by Tunisian users.

### 3. **Data Collection**
- We loop through the three apps and scrape their reviews.
- The reviews are stored in a list, with each app's data kept separately.

### 4. **Saving the Data**
- All reviews are combined into a single table (Pandas DataFrame).
- The data is saved as a CSV file (`tunisian_telco_reviews_raw.csv`) in the raw data folder.

---

## Output
- The notebook outputs a CSV file containing all the scraped reviews, ready for preprocessing and analysis.
- A summary of the collected data is displayed, including the first few rows for verification.

This notebook serves as the first step in the machine learning pipeline, providing the raw data needed for further processing and insights.

In [1]:
import pandas as pd                 # 'pandas' is like Excel for Python. It handles rows and columns.
from google_play_scraper import Sort, reviews_all  # The specialized tool to talk to Google Play Store.
from tqdm import tqdm               # Creates a progress bar so we know the code isn't frozen.
import os                           # Helps us work with computer folders (creating directories).

# ==========================================
# 2. CONFIGURATION (The Setup)
# ==========================================
# These are the "addresses" of the apps on the store. 
# I found these IDs by looking at the URL of the apps on play.google.com
APPS = {
    "Orange": "com.orange.myorange.otn",           # Orange Max it
    "Ooredoo": "tn.com.tunisiana.android.maTunisiana",     # My Ooredoo
    "MyTT": "com.tunisietelecom.selfcare"          # My TT (Tunisie Telecom)
}

# Where do we want to save the file?
# We use the folder structure we designed: data -> 01_raw (Raw means 'untouched')
RAW_DATA_FOLDER = "../data/01_raw/"
os.makedirs(RAW_DATA_FOLDER, exist_ok=True) # Create the folder if it doesn't exist

# ==========================================
# 3. THE ROBOT (The Function)
# ==========================================
def scrape_app(app_name, app_id):
    """
    This function takes an App Name and ID, goes to the store, 
    and comes back with a list of reviews.
    """
    print(f"Waking up robot to scrape: {app_name}...")
    
    try:
        # 'reviews_all' is the magic command. It fetches EVERYTHING.
        # lang='fr' and country='tn' ensures we see what a Tunisian user sees.
        result = reviews_all(
            app_id,
            sleep_milliseconds=0, # Go as fast as possible
            lang='fr',            # Language setting (French/Arabic mixed usually)
            country='tn',         # Store region: Tunisia
            sort=Sort.NEWEST,     # Get the newest reviews first
        )
        
        # Convert the raw result into a Pandas DataFrame (a Table)
        df = pd.DataFrame(result)
        
        # If we found reviews, let's tag them with the company name
        if not df.empty:
            df['company'] = app_name
            df['source'] = 'GooglePlay'
            
            # We only keep the columns we need.
            # 'content' = the text review
            # 'score' = 1 to 5 stars
            # 'at' = the date
            keep_cols = ['content', 'score', 'at', 'thumbsUpCount', 'company', 'source']
            df = df[keep_cols]
            
            print(f"Success! Copied {len(df)} reviews for {app_name}.")
            return df
        else:
            print(f"--- Robot came back empty for {app_name}.")
            return pd.DataFrame()
            
    except Exception as e:
        # If something breaks (like no internet), tell us why.
        print(f"--- Error scraping {app_name}: {e}")
        return pd.DataFrame()

# ==========================================
# 4. MAIN EXECUTION (Running the show)
# ==========================================
all_reviews = [] # An empty list to hold our piles of data

# Loop through our 3 apps (Orange, Ooredoo, MyTT)
for app_name, app_id in tqdm(APPS.items(), desc="Progress"):
    data = scrape_app(app_name, app_id)
    if not data.empty:
        all_reviews.append(data)

# ==========================================
# 5. SAVING (The Checkpoint)
# ==========================================
if all_reviews:
    # Glue all the separate app tables into one big table
    final_df = pd.concat(all_reviews, ignore_index=True)
    
    # Save to CSV (Comma Separated Values - readable by Excel)
    output_filename = "tunisian_telco_reviews_raw.csv"
    save_path = os.path.join(RAW_DATA_FOLDER, output_filename)
    final_df.to_csv(save_path, index=False)
    
    print("\n" + "="*40)
    print(f"MISSION ACCOMPLISHED!")
    print(f"Total Reviews Scraped: {len(final_df)}")
    print(f"Saved to: {save_path}")
    print("="*40)
    
    # Show the first 5 rows so we can see what we got
    print(final_df.head())

else:
    print("--- No data collected. Something went wrong.")

Progress:   0%|          | 0/3 [00:00<?, ?it/s]

Waking up robot to scrape: Orange...


Progress:  33%|███▎      | 1/3 [00:05<00:11,  5.51s/it]

Success! Copied 7289 reviews for Orange.
Waking up robot to scrape: Ooredoo...


Progress:  67%|██████▋   | 2/3 [00:14<00:07,  7.44s/it]

Success! Copied 12782 reviews for Ooredoo.
Waking up robot to scrape: MyTT...


Progress: 100%|██████████| 3/3 [00:16<00:00,  5.61s/it]

Success! Copied 3859 reviews for MyTT.

MISSION ACCOMPLISHED!
Total Reviews Scraped: 23930
Saved to: ../data/01_raw/tunisian_telco_reviews_raw.csv
                                             content  score  \
0                                  سراق محدش يستعمله      1   
1  des problèmes de démarrage répétitifs malgré l...      1   
2         Pretty sûre not a good experience, garbage      1   
3                             Je suis très satisfait      5   
4                                               bien      4   

                   at  thumbsUpCount company      source  
0 2025-12-03 12:29:20              0  Orange  GooglePlay  
1 2025-11-29 17:39:35              0  Orange  GooglePlay  
2 2025-11-25 15:28:43              0  Orange  GooglePlay  
3 2025-11-25 14:58:11              0  Orange  GooglePlay  
4 2025-11-22 23:32:20              0  Orange  GooglePlay  
